# Vertex Monitoring Book

This notebook contains all the details for assesing the vertex performance using the output of the `VertexMonitoringAlgorithm` provided in Pandora.

Not dependent on any ROOT/LArSoft specific tool, requires only the `.root` output files from the algorithm itself. 

In [2]:
import uproot

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt

import mplhep as hep
import hist

import glob
import pathlib
from pathlib import Path

from __future__ import annotations
from cycler import cycler

from typing import List, Optional

PlotStyle = {
    'axes.ymargin': 0.1,
    'legend.frameon': False,
    'xaxis.labellocation': 'right',
    'yaxis.labellocation': 'top',
    
    'axes.formatter.limits': (-2, 3),

    # 2. ax.ticklabel_format(useMathText=True)
    'axes.formatter.use_mathtext': True, 

    # 3. ax.minorticks_on()
    'xtick.minor.visible': True, 
    'ytick.minor.visible': True, 

    'xtick.major.size': 6, 
    'ytick.major.size': 6, 

    'xtick.labelsize': 14, 
    'ytick.labelsize': 14, 

    'xtick.direction': 'in', 
    'ytick.direction': 'in', 

    'xtick.top': True, 
    'ytick.right': True, 

    # 2. ax.tick_params(which='minor', length=3, direction='in', right=True, top=True)
    'xtick.minor.size': 3, 
    'ytick.minor.size': 3, 

    # Note: Direction, top, and right settings automatically apply to minor ticks 
    # when set globally, but you can explicitly ensure they mirror major ticks.
    'xtick.minor.top': True, 
    'ytick.minor.right': True, 

    'axes.xmargin': 0.0, 

    'legend.title_fontsize': 16, 
    'legend.fontsize': 14, 
    'axes.labelsize': 17, 
    'axes.titlesize': 16,

    'legend.handleheight': 1, 
    'legend.handlelength': 1.2, 

    'savefig.dpi': 300, 

    'axes.prop_cycle': (
        cycler('color', ["#0d49fb", "#e6091c", "#26eb47", "#8936df", "#fec32d", "#25d7fd"]) + 
        cycler('ls', ['-', '--', '-.', ':', '-', '--'])
    ), 
    'savefig.transparent': True,
    'savefig.bbox': 'tight',
}

# plt.style.use('dark_background')
plt.style.use(PlotStyle)

Below we define some "standards"

In [3]:
DATA_PATH = Path('../data')
SAVE_FORMATS = ['pdf', 'png', 'jpg']

def load_one(file, tree = 'vertices', preselection: Optional[str | List[str]] = None):
    with uproot.open(file) as f:
        names = [k for k in f.keys(cycle=False) if k.startswith(tree)]
        dfs = []
        for name in names:
            df = f[name].arrays(library="pd")
            df["tree"] = name

            if preselection:
                de = df.query(preselection)
            
            dfs.append(df)
    
    df = pd.concat(dfs, ignore_index=True)
    return df

def load(path: Path, preselection: Optional[str | List[str]] = None):
    '''Load the data from the .root files 
    in the `path`. If `preselection`,  
    return the skimmed version of the data

    @params
        path: required, Path object
            path to the data files
        preselection: str | List[str]
    '''

    for file in path.glob('*.root'):
        df = load_one(file, preselection=preselection)
    

# data_standard_file = sorted(DATA_PATH.glob('*_Standard.root'))[0]
# data_dl_file = sorted(DATA_PATH.glob('*_DL.root'))[0]

## Utilities + plotting goes here

Below there are a set of standard functions used for validation of the algorithm

In [73]:


def success_rate(data):
    return len(data[data.success == 1]) / len(data.success)

In [77]:
data_standard = load(data_standard_file)
data_dl = load(data_dl_file)